In [1]:
import pandas as pd, si_units as si, ternary, matplotlib.pyplot as plt, numpy as np, feos, sys
import TERNARY_CLASS as TC
sys.path.append("..")
import PLOT_SETTINGS as ps
from molmass import Formula

In [2]:
parameters      = feos.Parameters.from_json(["carbon dioxide", "hydrogen", "argon"], "../parameters.json")
T               = 220
# P               = 20* si.BAR
# n_total         = 1.0 * si.MOL
eos             = feos.HelmholtzEnergyFunctional.pcsaft(parameters)
parameters

|component|molarweight|m|sigma|epsilon_k|
|-|-|-|-|-|
|carbon dioxide|43.99|2.53096|2.57855|153.31864|
|hydrogen|2.016|1.0|3.002|51.343|
|argon|39.962|1.0|3.37751|117.80903|

In [3]:
feeds = np.array([0.95,0.01,0.04])
# feeds
CT, CP  = TC.compute_CT(feos, eos, feeds, T)
print(f"Critical Temperature: {CT / si.KELVIN:.2f} K")
print(f"Critical Pressure: {CP / si.BAR:.2f} bar")

Critical Temperature: 305.63 K
Critical Pressure: 89.19 bar


In [4]:
print(type(feeds))
print(feeds)

<class 'numpy.ndarray'>
[0.95 0.01 0.04]


In [5]:
T_values = range(210, int(CT / si.KELVIN) + 1)  # 210 K to critical temperature inclusive

bubble_pressures = []
dew_pressures = []
T_bubble = []
T_dew = []

TC.compute_bubble_curve(eos, T_values, feeds)

Bubble calculation failed at T = 298 K: Iteration resulted in trivial solution.
Bubble calculation failed at T = 299 K: Iteration resulted in trivial solution.
Bubble calculation failed at T = 304 K: Iteration resulted in trivial solution.
Bubble calculation failed at T = 305 K: Iteration resulted in trivial solution.


([210,
  211,
  212,
  213,
  214,
  215,
  216,
  217,
  218,
  219,
  220,
  221,
  222,
  223,
  224,
  225,
  226,
  227,
  228,
  229,
  230,
  231,
  232,
  233,
  234,
  235,
  236,
  237,
  238,
  239,
  240,
  241,
  242,
  243,
  244,
  245,
  246,
  247,
  248,
  249,
  250,
  251,
  252,
  253,
  254,
  255,
  256,
  257,
  258,
  259,
  260,
  261,
  262,
  263,
  264,
  265,
  266,
  267,
  268,
  269,
  270,
  271,
  272,
  273,
  274,
  275,
  276,
  277,
  278,
  279,
  280,
  281,
  282,
  283,
  284,
  285,
  286,
  287,
  288,
  289,
  290,
  291,
  292,
  293,
  294,
  295,
  296,
  297,
  300,
  301,
  302,
  303],
 [37.12434163233966,
  37.14868295233056,
  37.18150899620682,
  37.222879269396216,
  37.272856908288375,
  37.3315085511414,
  37.398904215399476,
  37.47511718104742,
  37.56022387939021,
  37.654303787138836,
  37.757439325185345,
  37.869715761843324,
  37.99122112018466,
  38.12204608914383,
  38.26228393800497,
  38.412030433984846,
  38.57138376

In [ ]:
for T_K in T_values:
    
    T = T_K * si.KELVIN
    try:

        envelope_vapor = feos.PhaseEquilibrium.dew_point(
            eos,
            temperature_or_pressure=T,
            vapor_molefracs=feeds)

        P_dew = envelope_vapor.vapor.pressure()

    except Exception as err:
        print(f"TP calculation failed at T = {T_K} K: {err}")
        break

    dew_pressures.append(P_dew/ si.BAR)
    T_dew.append(T/ si.KELVIN)

In [ ]:
plt.plot(T_bubble, bubble_pressures, 'b-', linewidth=2)
plt.plot(T_dew, dew_pressures, 'r-', linewidth=2)
plt.plot(CT / si.KELVIN, CP / si.BAR, 'ko', markersize=8, label='Critical Point')
plt.xlabel('Temperature (K)')
plt.ylabel('Pressure (bar)')
plt.grid(True)
plt.show()